# Diagnose FCCU + RFCCU Seasonal Anomaly

The dashboard's seasonal chart for `FCCU_UNIT_TYPES` shows ~58 mb/d offline in April
(2021–2025 band) and the same in Aug 2026E.  Global FCCU + RFCCU nameplate capacity
is roughly 22–25 mb/d, so the chart is reporting ~2.5× total fleet capacity offline
— physically impossible.

This notebook reuses the same data-loading and classification functions the Dash app
uses (`load_and_prepare`, `classify_events`, `monthly_capacity`) so any number we
print here matches what the chart would compute.  Each section tests one hypothesis
and ends with a `[PASS]`/`[FAIL]` verdict line.

**Read-only — no edits to `charts.py`, `data.py`, or `app.py`.**

In [1]:
import os
import sys

import numpy as np
import pandas as pd

# Make project modules importable when the notebook is launched from anywhere
PROJECT_ROOT = os.path.dirname(os.path.abspath('diagnose_fccu.ipynb'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import FCCU_UNIT_TYPES
from data import load_and_prepare
from classification import classify_events, enrich_events
from charts import monthly_capacity
from core import days_in_month

# Match the dashboard's "today vs previous working day" logic exactly
TODAY = pd.Timestamp.today().normalize()
_prev = TODAY - pd.Timedelta(days=1)
if _prev.weekday() >= 5:
    _prev -= pd.Timedelta(days=(_prev.weekday() - 4))
PREV = _prev

UTS = FCCU_UNIT_TYPES
print(f"Filter      : {UTS}")
print(f"Report date : {TODAY.date()}  (prev: {PREV.date()})")

print("\nLoading raw data from Snowflake …")
df_raw = load_and_prepare()
print(f"  df_raw     : {len(df_raw):,} rows")

print("Classifying …")
df_class = classify_events(df_raw, TODAY, PREV)
df_class = enrich_events(df_class, TODAY)
print(f"  df_class   : {len(df_class):,} unique events (latest snapshot per EVENT_ID)")



Filter      : ['FCCU (Fluid Catalytic Cracker)', 'RFCCU (Residue Fluid Catalytic Cracker)']
Report date : 2026-05-15  (prev: 2026-05-14)

Loading raw data from Snowflake …
  df_raw     : 738,557 rows
Classifying …
  df_class   : 169,187 unique events (latest snapshot per EVENT_ID)


In [2]:
df_class['UTYPE_DESC'].unique()

array(['Atmospheric Distillation', 'Delayed Coker',
       'FCCU Gasoline Hydrotreater', 'FCCU (Fluid Catalytic Cracker)',
       'Amine Treatment', 'Vacuum Distillation', 'Condensate Splitter',
       'Sulfur Recovery (SRU)', 'CCR (Continuous Catalytic Reformer)',
       'Distillate Hydrocracker', 'Sulfuric Alkylation',
       'Distillate Hydrotreater', 'Semiregen/Cyclic Reformer',
       'Mid Distillate Hydrotreater', 'Hydrofluoric Alkylation',
       'Visbreaker', 'Olefin/Aromatics Hydrotreater',
       'Diesel Hydrotreater', 'HGO (Heavy Gas Oils) Hydrotreater',
       'Reformate Splitter', 'Reformer Feed Hydrotreater',
       'FCCU Feed Hydrotreater', 'Propylene Splitter', 'Isomerization',
       'Asphalt', 'Lube Oil Hydrotreater', 'Naphtha Splitter',
       'Refinery Gas/Lights Ends', 'SR (Straight-Run) Hydrotreater',
       'Aromatics', 'Lube Oil Hydrocracker',
       'Diluent Recovery Unit (DRU)', 'Light Naphtha Hydrotreater',
       'Residual Hydrotreater', 'Hydrodealkylation',

In [10]:
UTS = ['FCCU (Fluid Catalytic Cracker)', 'RFCCU (Residue Fluid Catalytic Cracker)']
# Subset used by every hypothesis below
ut_col = 'UNIT_TYPE' if 'UNIT_TYPE' in df_class.columns else 'UTYPE_DESC'
df_fccu = df_class[df_class[ut_col].isin(UTS)].copy()
print(f"  df_fccu    : {len(df_fccu):,} FCCU/RFCCU events after filter")

  df_fccu    : 10,546 FCCU/RFCCU events after filter


## Cell 2 — Reproduce the chart number

Call `monthly_capacity` (the exact function `build_seasonal_chart` calls) for every
year × month and print the 2021–2026 grid in mb/d.  This anchors the rest of the
analysis: any later total we compute must reconcile with the cells flagged here.

A `THRESHOLD` of 30 mb/d is used below to highlight "obviously inflated" cells —
that's already ~1.3× plausible global FCCU + RFCCU offline capacity in any month.

In [11]:
YEARS = list(range(2021, 2027))
THRESHOLD = 30.0  # mb/d — anything above this is "implausible" and gets investigated below

grid = {}
for y in YEARS:
    grid[y] = monthly_capacity(df_class, y, unit_types=UTS)

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
chart_grid = pd.DataFrame(grid, index=months).T
chart_grid.index.name = 'Year'

print("Monthly avg offline capacity (mb/d) — exactly what the chart plots:\n")
display(chart_grid.round(2))

bad_cells = [(y, m) for y in YEARS for m, v in zip(months, grid[y]) if v >= THRESHOLD]
print(f"\nCells >= {THRESHOLD:.0f} mb/d: {len(bad_cells)}")
for y, m in bad_cells:
    print(f"  {y} {m}: {grid[y][months.index(m)]:.1f} mb/d")

Monthly avg offline capacity (mb/d) — exactly what the chart plots:



,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Year,,,,,,,,,,,,
2021,3.10,10.97,23.64,16.68,20.30,18.29,2.08,1.88,2.42,2.36,1.67,1.17
2022,1.60,1.73,38.03,58.18,20.10,2.01,1.75,1.47,1.71,2.33,1.71,1.43
2023,1.38,1.72,1.97,1.89,1.54,1.65,1.19,1.10,1.59,2.38,1.70,0.93
2024,1.21,1.91,1.57,1.91,1.74,1.65,1.63,1.39,1.81,1.73,1.21,0.71
2025,0.97,1.40,1.56,2.31,2.31,1.76,1.59,1.62,2.12,2.29,2.06,1.37
2026,1.19,1.43,1.44,1.87,1.51,1.03,22.41,56.44,10.29,1.10,0.57,0.23



Cells >= 30 mb/d: 3
  2022 Mar: 38.0 mb/d
  2022 Apr: 58.2 mb/d
  2026 Aug: 56.4 mb/d


## Cell 4 — Hypothesis A: filter over-matching

If `UNIT_TYPE` has near-duplicate strings (e.g. `"FCCU"`, `"FCC Unit"`, `"Fluid
Catalytic Cracker"`) and only some are in `FCCU_UNIT_TYPES`, the filter would *under*
match — not over-match — so this is the least likely culprit.  But verify anyway:
list every distinct `UNIT_TYPE` value that contains "FCC" (case-insensitive) and
confirm only the two strings we expect survive the filter.

In [5]:
all_uts = df_class[ut_col].dropna().unique()
fcc_like = sorted([u for u in all_uts if 'fcc' in str(u).lower() or 'catalytic' in str(u).lower()])

print(f"Distinct UNIT_TYPE values containing 'FCC' or 'catalytic' ({len(fcc_like)}):")
for u in fcc_like:
    in_filter = u in UTS
    marker = '  IN FILTER  ' if in_filter else '  ignored    '
    n = (df_class[ut_col] == u).sum()
    print(f"  {marker} {u!r:60s}  ({n:,} events)")

print("\nSurvivors after filter (should match the two strings above marked IN FILTER):")
print(df_fccu[ut_col].value_counts().to_string())

# Verdict
matched = set(df_fccu[ut_col].unique())
expected = set(UTS)
verdict = "[PASS]" if matched.issubset(expected) and matched == expected else "[FAIL]"
print(f"\n{verdict} filter matched exactly {sorted(matched)}; expected {sorted(expected)}")

Distinct UNIT_TYPE values containing 'FCC' or 'catalytic' (6):
    ignored     'CCR (Continuous Catalytic Reformer)'                         (8,033 events)
    IN FILTER   'FCCU (Fluid Catalytic Cracker)'                              (10,030 events)
    ignored     'FCCU Feed Hydrotreater'                                      (911 events)
    ignored     'FCCU Gasoline Hydrotreater'                                  (5,152 events)
    IN FILTER   'RFCCU (Residue Fluid Catalytic Cracker)'                     (516 events)
    ignored     'TCCU (Thermal Catalytic Cracking Unit)'                      (43 events)

Survivors after filter (should match the two strings above marked IN FILTER):
UTYPE_DESC
FCCU (Fluid Catalytic Cracker)             10030
RFCCU (Residue Fluid Catalytic Cracker)      516

[PASS] filter matched exactly ['FCCU (Fluid Catalytic Cracker)', 'RFCCU (Residue Fluid Catalytic Cracker)']; expected ['FCCU (Fluid Catalytic Cracker)', 'RFCCU (Residue Fluid Catalytic Cracker)']


## Cell 6 — Hypothesis B: mixed UOM in `CAP_OFFLINE_STD`

`CONVERSION_FACTORS` in `data.py` maps `BBL/d → kbd` but `M Lbs/yr` and `Metric T/yr
→ kT/d`.  The chart sums `CAP_OFFLINE_STD` blindly — if any FCCU row arrives with a
non-volumetric UOM, the chart is adding kbd + kT/d as if they were the same unit.

For FCCU/RFCCU we expect every row to be `BBL/d` (volumetric throughput).  Anything
else gets printed individually for inspection.

In [12]:
uom_in = next((c for c in ['CAP_UOM', 'UOM'] if c in df_fccu.columns), None)
uom_out = 'CAP_UOM_STD' if 'CAP_UOM_STD' in df_fccu.columns else None

print(f"Source UOM column      : {uom_in}")
print(f"Standardized UOM column: {uom_out}\n")

if uom_in:
    print("Source UOM distribution (FCCU/RFCCU only):")
    print(df_fccu[uom_in].value_counts(dropna=False).to_string())

if uom_out:
    print("\nStandardized UOM distribution:")
    print(df_fccu[uom_out].value_counts(dropna=False).to_string())

# Anything not BBL/d is suspicious for an FCCU
suspect = df_fccu[~df_fccu[uom_in].isin(['BBL/d'])] if uom_in else pd.DataFrame()
print(f"\nRows with non-BBL/d UOM: {len(suspect)}")
if not suspect.empty:
    show_cols = [c for c in ['EVENT_ID','PLANT_NAME','UNIT_NAME','UNIT_TYPE',
                              uom_in,'CAP_OFFLINE','CAP_OFFLINE_STD'] if c in suspect.columns]
    display(suspect[show_cols].head(20))

verdict = "[PASS]" if suspect.empty else f"[FAIL] {len(suspect)} rows with non-BBL/d UOM mixed into CAP_OFFLINE_STD"
print(f"\n{verdict}")

Source UOM column      : CAP_UOM
Standardized UOM column: CAP_UOM_STD

Source UOM distribution (FCCU/RFCCU only):
CAP_UOM
BBL/d    10519
None        27

Standardized UOM distribution:
CAP_UOM_STD
kbd     10519
None       27

Rows with non-BBL/d UOM: 27


,EVENT_ID,PLANT_NAME,UNIT_NAME,CAP_UOM,CAP_OFFLINE,CAP_OFFLINE_STD
7857,269352,Hainan Refinery,RFCCU,None,56000,56000.0
10368,286617,Yueyang Refinery (Sinopec Changling),RFCCU 1,None,20000,20000.0
21736,359828,Dushanzi Refinery,RFCCU 2,None,12000,12000.0
25660,378852,Yueyang Refinery (Sinopec Changling),RFCCU 1,None,20000,20000.0
28695,388125,Dushanzi Refinery,RFCCU 2,None,None,NaN
28730,388254,Dushanzi Refinery,RFCCU 2,None,12000,12000.0
32606,402135,Dushanzi Refinery,RFCCU 2,None,None,NaN
52771,462096,Hainan Refinery,RFCCU,None,28000,28000.0
70917,506821,Yueyang Refinery (Sinopec Changling),RFCCU 1,None,6000,6000.0
71836,508762,Dushanzi Refinery,RFCCU 2,None,2880,2880.0



[FAIL] 27 rows with non-BBL/d UOM mixed into CAP_OFFLINE_STD


## Cell 8 — Hypothesis C: open-ended / extra-long events

The strongest suspect.  `charts.py` line 37:

```python
ev['_end'] = ev['END_DATE'].fillna(yr_end).clip(upper=yr_end)
```

Any event with a missing `END_DATE` is silently treated as active through Dec 31 of
**every year** the chart evaluates.  A single 200 kbd FCCU with `END_DATE = NaT`
contributes its full capacity to every month of 2021, 2022, …, 2026.

Similarly, events with multi-year `END_DATE`s contribute to every month they span.
We list both populations here and sum their capacity — if the total approaches the
~33 mb/d gap between physical reality (~22–25 mb/d) and the chart (~58 mb/d), this
is the root cause.

In [13]:
# Optionally exclude cancelled events to mirror monthly_capacity()
_active = df_fccu.copy()
if 'E_STATUS_RESEARCH' in _active.columns:
    _active = _active[_active['E_STATUS_RESEARCH'] != 'Cancelled']

cap = pd.to_numeric(_active['CAP_OFFLINE_STD'], errors='coerce').fillna(0)
_active = _active.assign(_cap_kbd=cap)

# --- (1) Missing END_DATE ---
miss_end = _active[_active['END_DATE'].isna() & _active['START_DATE'].notna()].copy()
miss_end = miss_end.sort_values('_cap_kbd', ascending=False)

show_cols = [c for c in ['EVENT_ID','PLANT_NAME','UNIT_NAME','COUNTRY','UNIT_TYPE',
                          'START_DATE','END_DATE','_cap_kbd','E_STATUS_RESEARCH','EVENT_TYPE']
             if c in miss_end.columns]

print(f"FCCU/RFCCU events with missing END_DATE: {len(miss_end)}")
print(f"  Sum of CAP_OFFLINE_STD: {miss_end['_cap_kbd'].sum():,.0f} kbd  "
      f"= {miss_end['_cap_kbd'].sum()/1000:,.2f} mb/d\n")
if not miss_end.empty:
    display(miss_end[show_cols].head(20))

# --- (2) Very long events (END_DATE - START_DATE > 365 days) ---
long_dur = _active.dropna(subset=['START_DATE','END_DATE']).copy()
long_dur['_dur_days'] = (long_dur['END_DATE'] - long_dur['START_DATE']).dt.days
long_dur = long_dur[long_dur['_dur_days'] > 365].sort_values('_cap_kbd', ascending=False)

print(f"\nFCCU/RFCCU events with duration > 365 days: {len(long_dur)}")
print(f"  Sum of CAP_OFFLINE_STD: {long_dur['_cap_kbd'].sum():,.0f} kbd  "
      f"= {long_dur['_cap_kbd'].sum()/1000:,.2f} mb/d\n")
if not long_dur.empty:
    show2 = show_cols + ['_dur_days']
    show2 = [c for c in show2 if c in long_dur.columns]
    display(long_dur[show2].head(20))

# --- (3) Combined population active outside its "natural" year ---
suspect_total = (miss_end['_cap_kbd'].sum() + long_dur['_cap_kbd'].sum()) / 1000
print(f"\nCombined inflated capacity carried across years: {suspect_total:.2f} mb/d")

gap = 58 - 24  # observed (~58 mb/d) vs plausible upper bound (~24 mb/d global FCCU)
verdict = "[FAIL]" if suspect_total >= 0.5 * gap else "[PASS]"
print(f"{verdict} this category {'explains' if verdict=='[FAIL]' else 'does NOT explain'} "
      f"a large share of the ~{gap:.0f} mb/d gap")

FCCU/RFCCU events with missing END_DATE: 0
  Sum of CAP_OFFLINE_STD: 0 kbd  = 0.00 mb/d


FCCU/RFCCU events with duration > 365 days: 68
  Sum of CAP_OFFLINE_STD: 1,453 kbd  = 1.45 mb/d



,EVENT_ID,PLANT_NAME,UNIT_NAME,COUNTRY,START_DATE,END_DATE,_cap_kbd,E_STATUS_RESEARCH,EVENT_TYPE,_dur_days
20974,356061,Ruwais Refinery 2 (West),RFCCU (U1600),United Arab Emirates,2017-01-11,2019-01-24,127.2000,Past,Unplanned,743
7537,267138,Torrance Refinery,FCCU,U.S.A.,2015-02-18,2016-05-10,100.0000,Past,Unplanned,447
15696,322308,El Palito Refinery,FCCU,Venezuela,2018-12-01,2020-04-12,70.0000,Past,Planned,498
143939,665199,El Palito Refinery,FCCU,Venezuela,2022-05-15,2023-06-09,70.0000,Past,Unplanned,390
161662,767240,Martinez Refinery (PBF),CP Catalytic Cracking Unit (CCU),U.S.A.,2025-02-01,2026-04-26,70.0000,Past,Unplanned,449
70070,504654,Willemstad Isla Curacao Refinery,FCCU,Curacao,2020-04-27,2021-12-01,54.0000,Past,Unplanned,583
213,88928,Willemstad Isla Curacao Refinery,FCCU,Curacao,2018-02-28,2020-01-05,54.0000,Past,Planned,676
31666,399035,Port Harcourt Refinery (NNPC),Fluid Catalytic Cracker Unit,Nigeria,2021-12-15,2026-03-15,40.0000,Past,Planned,1551
47609,449253,Port Harcourt Refinery (NNPC),Fluid Catalytic Cracker Unit,Nigeria,2018-09-24,2021-12-14,40.0000,Past,Unplanned,1177
31651,399010,Port Harcourt Refinery (NNPC),Fluid Catalytic Cracker Unit,Nigeria,2016-10-15,2017-11-27,40.0000,Past,Unplanned,408



Combined inflated capacity carried across years: 1.45 mb/d
[PASS] this category does NOT explain a large share of the ~34 mb/d gap


## Cell 10 — Hypothesis D: duplicate physical units

`classify_events()` dedupes by `EVENT_ID` only.  If the same physical FCCU has
multiple `EVENT_ID`s with overlapping date ranges (e.g. planned TA + unplanned trip
logged separately, or IIR re-issuing an event under a new ID), both contribute their
full capacity to every day they overlap.

For each `(PLANT_NAME, UNIT_NAME)` pair, find days where ≥2 of its events are
simultaneously active and report the worst offenders.

In [45]:
window_start = pd.Timestamp(2021, 1, 1)
window_end   = pd.Timestamp(2026, 12, 31)

# Use the same "_end fill" semantics the chart uses so we compare apples-to-apples
dup_src = _active.copy()
dup_src['_end_filled'] = dup_src['END_DATE'].fillna(window_end).clip(upper=window_end)
dup_src['_start_clip'] = dup_src['START_DATE'].clip(lower=window_start)
dup_src = dup_src[(dup_src['_start_clip'] <= window_end) & (dup_src['_end_filled'] >= window_start)]

key_cols = [c for c in ['PLANT_NAME', 'UNIT_NAME'] if c in dup_src.columns]
if not key_cols:
    print("[SKIP] no PLANT_NAME/UNIT_NAME columns to group on")
else:
    overlap_rows = []
    for keys, grp in dup_src.groupby(key_cols):
        if len(grp) < 2:
            continue
        rows = grp[['EVENT_ID', '_start_clip', '_end_filled', '_cap_kbd']].sort_values('_start_clip').to_records(index=False)
        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                # Days the two events were simultaneously active
                ov_start = max(rows[i]._start_clip, rows[j]._start_clip)
                ov_end   = min(rows[i]._end_filled, rows[j]._end_filled)
                # to_records() makes the timestamps numpy.datetime64, whose timedelta
                # has no .days — divide by np.timedelta64(1,'D') to get days as a float.
                ov_days  = int((ov_end - ov_start) / np.timedelta64(1, 'D')) + 1
                if ov_days <= 0:
                    continue
                # Inflated kbd is the smaller of the two capacities (the duplicate "extra")
                extra_kbd = min(rows[i]._cap_kbd, rows[j]._cap_kbd)
                overlap_rows.append({
                    'plant_unit': keys if isinstance(keys, tuple) else (keys,),
                    'event_a': rows[i].EVENT_ID, 'event_b': rows[j].EVENT_ID,
                    'overlap_days': ov_days,
                    'extra_kbd': extra_kbd,
                    'inflated_kbd_days': extra_kbd * ov_days,
                })

    if not overlap_rows:
        print("[PASS] no overlapping EVENT_IDs at any (plant, unit) level")
    else:
        ov = pd.DataFrame(overlap_rows).sort_values('inflated_kbd_days', ascending=False)
        # Total inflated capacity-days, normalised to mb/d averaged over the 6-year window
        window_days = (window_end - window_start).days + 1
        avg_inflation_mbd = ov['inflated_kbd_days'].sum() / window_days / 1000
        print(f"Overlap pairs found: {len(ov)}")
        print(f"Avg inflation across 2021-2026: {avg_inflation_mbd:.2f} mb/d "
              f"(spread over {window_days:,} days)")
        print("\nTop 20 worst overlaps (by inflated kbd·days):")
        display(ov.head(20))

        verdict = "[FAIL]" if avg_inflation_mbd >= 2 else "[PASS]"
        print(f"\n{verdict} duplicate-event overlap contributes "
              f"{avg_inflation_mbd:.2f} mb/d on average")

Overlap pairs found: 3
Avg inflation across 2021-2026: 0.00 mb/d (spread over 2,191 days)

Top 20 worst overlaps (by inflated kbd·days):


,plant_unit,event_a,event_b,overlap_days,extra_kbd,inflated_kbd_days
0,"(Lekki Refinery (Dangote), Residue Fluidized C...",789867,795392,47,81.60,3835.2
1,"(Lekki Refinery (Dangote), Residue Fluidized C...",789867,795274,3,204.00,612.0
2,"(Zarqa Refinery, Fluid Catalytic Cracker Unit)",774852,796269,42,1.35,56.7



[PASS] duplicate-event overlap contributes 0.00 mb/d on average


## Cell 12 — Hypothesis E: per-event capacity > unit nameplate

If `CAP_OFFLINE_STD` exceeds the unit's own `U_CAPACITY_STD`, the event is reporting
more offline capacity than the unit physically has — almost always because IIR
encoded a *site-wide* outage against a single unit row.  We flag any ratio > 1.05
(small slack for rounding).

In [15]:
if 'U_CAPACITY_STD' not in df_fccu.columns:
    print("[SKIP] no U_CAPACITY_STD column available")
else:
    r = df_fccu.copy()
    r['_cap_off'] = pd.to_numeric(r['CAP_OFFLINE_STD'], errors='coerce')
    r['_cap_nom'] = pd.to_numeric(r['U_CAPACITY_STD'], errors='coerce')
    r['ratio']   = r['_cap_off'] / r['_cap_nom']

    flagged = r[(r['ratio'] > 1.05) & r['_cap_nom'].notna() & (r['_cap_nom'] > 0)].copy()
    flagged = flagged.sort_values('ratio', ascending=False)

    show = [c for c in ['EVENT_ID','PLANT_NAME','UNIT_NAME','COUNTRY','UNIT_TYPE',
                         'START_DATE','END_DATE','_cap_off','_cap_nom','ratio',
                         'E_STATUS_RESEARCH','EVENT_TYPE'] if c in flagged.columns]

    print(f"FCCU/RFCCU events with CAP_OFFLINE_STD > U_CAPACITY_STD * 1.05: {len(flagged)}")
    print(f"Excess capacity beyond nameplate: "
          f"{(flagged['_cap_off'] - flagged['_cap_nom']).sum():,.0f} kbd  "
          f"= {(flagged['_cap_off'] - flagged['_cap_nom']).sum()/1000:,.2f} mb/d")

    if not flagged.empty:
        display(flagged[show].head(20))

    verdict = "[FAIL]" if not flagged.empty else "[PASS]"
    print(f"\n{verdict} {'some events list more offline capacity than the unit itself' if not flagged.empty else 'no over-nameplate offline capacities'}")

FCCU/RFCCU events with CAP_OFFLINE_STD > U_CAPACITY_STD * 1.05: 0
Excess capacity beyond nameplate: 0 kbd  = 0.00 mb/d

[PASS] no over-nameplate offline capacities


## Cell 14 — Top contributors per "bad" month

For every `(year, month)` cell from Cell 3 above `THRESHOLD` mb/d, replicate the
`monthly_capacity` overlap math inline — but keep per-event contributions instead
of summing — and show the top 15.  This is the definitive list of *which events*
are driving the inflated bars, no matter which hypothesis above is the cause.

The full table is also saved to `diagnostics/fccu_top_contributors_<YYYYMMDD>.csv`
for offline review.

In [16]:
def per_event_contributions(df, year, month):
    """Replicate monthly_capacity() but return one row per event with its
    mb/d contribution to that (year, month) cell."""
    ms = pd.Timestamp(year, month, 1)
    me = pd.Timestamp(year, month, days_in_month(year, month))
    d  = (me - ms).days + 1
    yr_end = pd.Timestamp(year, 12, 31)

    ev = df.copy()
    if 'E_STATUS_RESEARCH' in ev.columns:
        ev = ev[ev['E_STATUS_RESEARCH'] != 'Cancelled']
    ev['_end'] = ev['END_DATE'].fillna(yr_end).clip(upper=yr_end)
    ev = ev[ev['START_DATE'].notna() & (ev['START_DATE'] <= yr_end)]
    ev = ev[ev['_end'] >= pd.Timestamp(year, 1, 1)]
    act = ev[(ev['START_DATE'] <= me) & (ev['_end'] >= ms)].copy()
    if act.empty:
        return pd.DataFrame()

    overlap = (act['_end'].clip(upper=me) - act['START_DATE'].clip(lower=ms)).dt.days + 1
    cap = pd.to_numeric(act['CAP_OFFLINE_STD'], errors='coerce').fillna(0)
    act = act.assign(
        overlap_days = overlap.clip(lower=0),
        contribution_mbd = (cap * overlap.clip(lower=0) / d) / 1000,
        days_in_month = d,
    )
    return act.sort_values('contribution_mbd', ascending=False)


cells_to_audit = [(y, mi + 1) for y in YEARS for mi, v in enumerate(grid[y]) if v >= THRESHOLD]
print(f"Auditing {len(cells_to_audit)} cell(s) >= {THRESHOLD:.0f} mb/d: {cells_to_audit}\n")

show = ['EVENT_ID','PLANT_NAME','UNIT_NAME','COUNTRY','UNIT_TYPE',
        'EVENT_TYPE','E_STATUS_RESEARCH',
        'START_DATE','END_DATE','CAP_OFFLINE_STD',
        'overlap_days','days_in_month','contribution_mbd']

all_contribs = []
for y, m in cells_to_audit:
    contribs = per_event_contributions(df_fccu, y, m)
    if contribs.empty:
        continue
    contribs = contribs.assign(year=y, month=months[m-1])
    all_contribs.append(contribs)
    cols_avail = [c for c in show if c in contribs.columns]
    print(f"\n=== {y} {months[m-1]}  total={grid[y][m-1]:.1f} mb/d  "
          f"({len(contribs)} active events) ===")
    display(contribs[cols_avail].head(15))

if all_contribs:
    out_dir = os.path.join(PROJECT_ROOT, 'diagnostics')
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"fccu_top_contributors_{TODAY.strftime('%Y%m%d')}.csv")
    full = pd.concat(all_contribs, ignore_index=True)
    # Lighten the CSV — drop large/sensitive columns that don't help triage
    drop = [c for c in ['COMMENTS','SOURCE','PARENTNAME','TIME_STAMP','RELEASE_DT',
                         'LIVE_DATE','_end','_dur_now','_dur_prev','_fwd_days',
                         '_fwd_days_prev']
            if c in full.columns]
    full.drop(columns=drop).to_csv(out_path, index=False)
    print(f"\nFull contributor table saved -> {out_path}  ({len(full):,} rows)")
else:
    print("No cells exceed threshold; nothing saved.")

Auditing 3 cell(s) >= 30 mb/d: [(2022, 3), (2022, 4), (2026, 8)]


=== 2022 Mar  total=38.0 mb/d  (169 active events) ===


,EVENT_ID,PLANT_NAME,UNIT_NAME,COUNTRY,EVENT_TYPE,E_STATUS_RESEARCH,START_DATE,END_DATE,CAP_OFFLINE_STD,overlap_days,days_in_month,contribution_mbd
74608,513989,Hainan Refinery,RFCCU,China,Planned,Past,2022-03-12,2022-05-10,56000.0,20,31,36.129032
90975,544683,Baton Rouge Refinery,PCLA 3 (FCCU),U.S.A.,Planned,Past,2022-02-24,2022-04-11,110.0,31,31,0.110000
121381,609435,Mailiao Refinery,Resid Fluid Cat Cracker 02,Republic of China-Taiwan,Planned,Past,2022-03-01,2022-04-29,84.0,31,31,0.084000
5220,239220,Cardon Refinery,FCCU,Venezuela,Planned,Past,2021-07-11,2022-05-10,77.0,31,31,0.077000
90183,543282,Los Angeles Refinery - Carson (Marathon),FCCU,U.S.A.,Planned,Past,2022-03-11,2022-04-14,104.0,21,31,0.070452
134438,639917,RU VI - Balongan Refinery,RCCU,Indonesia,Planned,Past,2022-03-07,2022-05-21,83.0,25,31,0.066935
124256,616013,Corpus Christi Refinery - East (CITGO),FCCU 2,U.S.A.,Planned,Past,2022-01-17,2022-03-28,69.0,28,31,0.062323
136365,644128,Bilbao Refinery,FCCU (F3),Spain,Planned,Past,2022-02-13,2022-04-17,46.0,31,31,0.046000
12023,295540,Ufa Oil Refinery,Catalytic Cracker G-43-107/M1 (C-200),Russia,Planned,Past,2022-02-10,2022-04-18,40.0,31,31,0.040000
31666,399035,Port Harcourt Refinery (NNPC),Fluid Catalytic Cracker Unit,Nigeria,Planned,Past,2021-12-15,2026-03-15,40.0,31,31,0.040000



=== 2022 Apr  total=58.2 mb/d  (140 active events) ===


,EVENT_ID,PLANT_NAME,UNIT_NAME,COUNTRY,EVENT_TYPE,E_STATUS_RESEARCH,START_DATE,END_DATE,CAP_OFFLINE_STD,overlap_days,days_in_month,contribution_mbd
74608,513989,Hainan Refinery,RFCCU,China,Planned,Past,2022-03-12,2022-05-10,56000.0,30,30,56.000000
134438,639917,RU VI - Balongan Refinery,RCCU,Indonesia,Planned,Past,2022-03-07,2022-05-21,83.0,30,30,0.083000
121381,609435,Mailiao Refinery,Resid Fluid Cat Cracker 02,Republic of China-Taiwan,Planned,Past,2022-03-01,2022-04-29,84.0,29,30,0.081200
5220,239220,Cardon Refinery,FCCU,Venezuela,Planned,Past,2021-07-11,2022-05-10,77.0,30,30,0.077000
142540,660161,El Palito Refinery,FCCU,Venezuela,Unplanned,Past,2022-04-01,2022-05-06,70.0,30,30,0.070000
140942,655180,Joliet Refinery,FCCU,U.S.A.,Planned,Past,2022-03-21,2022-04-19,98.0,19,30,0.062067
90183,543282,Los Angeles Refinery - Carson (Marathon),FCCU,U.S.A.,Planned,Past,2022-03-11,2022-04-14,104.0,14,30,0.048533
90975,544683,Baton Rouge Refinery,PCLA 3 (FCCU),U.S.A.,Planned,Past,2022-02-24,2022-04-11,110.0,11,30,0.040333
31666,399035,Port Harcourt Refinery (NNPC),Fluid Catalytic Cracker Unit,Nigeria,Planned,Past,2021-12-15,2026-03-15,40.0,30,30,0.040000
18644,342436,Bakinsk Heydar Aliyev Upper Refinery,Catalytic Cracker 1A/1M,Azerbaijan,Planned,Past,2022-03-30,2022-05-25,40.0,30,30,0.040000



=== 2026 Aug  total=56.4 mb/d  (22 active events) ===


,EVENT_ID,PLANT_NAME,UNIT_NAME,COUNTRY,EVENT_TYPE,E_STATUS_RESEARCH,START_DATE,END_DATE,CAP_OFFLINE_STD,overlap_days,days_in_month,contribution_mbd
134213,639206,Hainan Refinery,RFCCU,China,Planned,Future,2026-07-20,2026-09-05,56000.0,31,31,56.000000
14414,311861,Hohhot Refinery,Fluid Catalytic Cracker 2,China,Planned,Future,2026-07-05,2026-08-28,56.0,28,31,0.050581
125806,618956,Luqing Refinery,RFCCU,China,Planned,Future,2026-07-15,2026-08-30,42.0,30,31,0.040645
145807,673720,Yan'an Refinery,FCCU 3,China,Planned,Future,2026-07-15,2026-09-02,40.0,31,31,0.040000
168529,814492,Al Jubail SATORP Refinery,FCCU (U061),Saudi Arabia,Unplanned,Ongoing,2026-04-07,2026-09-30,37.5,31,31,0.037500
161333,763994,Lytton Refinery,FCCU,Australia,Planned,Future,2026-08-02,2026-09-11,35.0,30,31,0.033871
144695,668088,Dongying Refinery (Shandong Wantong),DCC,China,Planned,Future,2026-07-10,2026-08-28,36.0,28,31,0.032516
160079,753337,Qingyang Refinery,FCCU 1,China,Planned,Future,2026-07-25,2026-09-10,32.0,31,31,0.032000
33756,405995,Karamay Refinery,FCCU,China,Planned,Future,2026-08-01,2026-09-19,20.0,31,31,0.020000
145810,673724,Yan'an Refinery,FCCU 2,China,Planned,Future,2026-07-15,2026-09-02,20.0,31,31,0.020000



Full contributor table saved -> c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\dash_irr_database\diagnostics\fccu_top_contributors_20260515.csv  (331 rows)


## Cell 16 — Fleet capacity reality check

Sum `U_CAPACITY_STD` over distinct `(PLANT_NAME, UNIT_NAME)` pairs for FCCU/RFCCU.
This is the **upper bound** on physically possible offline capacity at any moment,
because the most that can be offline is the entire fleet being shut down at once.

The chart's worst cell should not exceed this number.  If it does, the inflation
is mathematically certain and not a question of interpretation.

In [17]:
cap_col_nom = 'U_CAPACITY_STD' if 'U_CAPACITY_STD' in df_raw.columns else None
ut_col_raw  = 'UNIT_TYPE'      if 'UNIT_TYPE'      in df_raw.columns else 'UTYPE_DESC'

if cap_col_nom is None:
    print("[SKIP] no U_CAPACITY_STD in df_raw")
else:
    fleet = (df_raw[df_raw[ut_col_raw].isin(UTS)]
             .dropna(subset=['PLANT_NAME','UNIT_NAME'])
             .sort_values('TIME_STAMP')
             .groupby(['PLANT_NAME','UNIT_NAME'])
             .last()
             .reset_index())

    fleet_kbd = pd.to_numeric(fleet[cap_col_nom], errors='coerce').fillna(0).sum()
    fleet_mbd = fleet_kbd / 1000

    print(f"Distinct FCCU/RFCCU units in dataset : {len(fleet):,}")
    print(f"Sum of nameplate capacity (kbd)      : {fleet_kbd:,.0f}")
    print(f"Sum of nameplate capacity (mb/d)     : {fleet_mbd:,.2f}")
    print(f"Chart's worst month value (mb/d)     : {chart_grid.values.max():,.2f}")
    print(f"Ratio chart / fleet                  : {chart_grid.values.max() / fleet_mbd:.2f}x")

    if chart_grid.values.max() > fleet_mbd:
        print("\n[FAIL] chart exceeds total fleet capacity — inflation is mathematically certain")
    else:
        print("\n[PASS] chart is within fleet upper bound")

Distinct FCCU/RFCCU units in dataset : 590
Sum of nameplate capacity (kbd)      : 110,889
Sum of nameplate capacity (mb/d)     : 110.89
Chart's worst month value (mb/d)     : 58.18
Ratio chart / fleet                  : 0.52x

[PASS] chart is within fleet upper bound


## Step 1 — What-if: patch the missing-UOM rows

Hypothesis B was confirmed: ~27 events have a missing `CAP_UOM`.  In
[`data.py`](data.py) `apply_conversion()`, the line

```python
df.loc[unmapped, out_cap] = cap_num[unmapped]
```

copies the raw `CAP_OFFLINE` value (typically in `BBL/d`) straight into
`CAP_OFFLINE_STD` with **no scaling**, so volumetric values are inflated by 1000×.

We test the proposed fix here **without modifying any production code**.

**Rule we're testing** (your idea, inverted to be maintainable):
> For rows with missing UOM, default to `BBL/d` *unless* `UNIT_TYPE` is in a known
> non-volumetric set — believed to be `{Hydrodealkylation, Propylene Splitter,
> Sulfur Recovery (SRU)}`.

The cell first **verifies** that set from `df_raw` (every UNIT_TYPE that ever has a
non-BBL/d UOM), then patches a *copy* of `df_class` and re-renders the seasonal
chart so we can compare before/after side by side.

In [27]:
view = df_raw.groupby(['UTYPE_DESC']).last()

In [31]:
view[['CAP_UOM']]

,CAP_UOM
UTYPE_DESC,
Amine Treatment,MMSCFD
Aromatics,BBL/d
Asphalt,BBL/d
Atmospheric Distillation,BBL/d
Benzene,Metric T/yr
CCR (Continuous Catalytic Reformer),BBL/d
CCR(Continuous Catalytic Reformer),BBL/d
Condensate Splitter,BBL/d
Condensate Stabilizer,None


In [18]:
from charts import build_seasonal_chart

# User's hypothesis: only these 3 UNIT_TYPEs are non-volumetric.
# We default missing-UOM rows to BBL/d for every OTHER unit type.
NON_VOLUMETRIC_UTS = {
    'Hydrodealkylation',
    'Propylene Splitter',
    'Sulfur Recovery (SRU)',
}



uom_col_raw = 'CAP_UOM' if 'CAP_UOM' in df_raw.columns else 'UOM'

# --- (1) Verify the non-volumetric set against actual data ---
# Build a UNIT_TYPE × UOM count pivot, then keep only types that ever use
# something other than BBL/d.  Anything that shows up here MUST be in
# NON_VOLUMETRIC_UTS or we risk corrupting its values with a BBL/d default.
pivot = (df_raw.dropna(subset=[ut_col_raw, uom_col_raw])
                .groupby([ut_col_raw, uom_col_raw]).size()
                .unstack(fill_value=0))
non_bbld_cols = [c for c in pivot.columns if 'bbl' not in str(c).lower()]
suspicious = (pivot[pivot[non_bbld_cols].sum(axis=1) > 0]
              if non_bbld_cols else pivot.iloc[:0])

print(f"UNIT_TYPEs that EVER use a non-BBL/d UOM ({len(suspicious)}):")
display(suspicious)

found = set(suspicious.index)
extra   = found - NON_VOLUMETRIC_UTS    # in data but NOT in our list — would be wrongly defaulted
missing = NON_VOLUMETRIC_UTS - found    # in our list but not seen in data — harmless

if not extra:
    print(f"\n[PASS] every non-volumetric UNIT_TYPE in the data is in NON_VOLUMETRIC_UTS")
else:
    print(f"\n[REVIEW] UNIT_TYPEs with non-BBL/d UOM that are NOT in your exclusion list:")
    print(f"         {extra}")
    print("         → add them to NON_VOLUMETRIC_UTS before applying the real fix.")
if missing:
    print(f"[INFO] in your list but not seen with non-BBL/d UOM in this data: {missing}")


# --- (2) Inspect the missing-UOM rows we're about to patch ---
missing_uom_mask = (df_class[uom_col_raw].isna()
                    | (df_class[uom_col_raw].astype(str).str.strip() == ''))
missing_rows = df_class[missing_uom_mask].copy()
print(f"\nEvents in df_class with missing UOM: {len(missing_rows)}")
if not missing_rows.empty:
    show_cols = [c for c in ['EVENT_ID','PLANT_NAME','UNIT_NAME',ut_col,
                              'CAP_OFFLINE','CAP_OFFLINE_STD','U_CAPACITY','U_CAPACITY_STD',
                              'START_DATE','END_DATE','E_STATUS_RESEARCH']
                 if c in missing_rows.columns]
    display(missing_rows[show_cols].head(40))
    print("\nDistribution of those rows by UNIT_TYPE:")
    print(missing_rows[ut_col].value_counts(dropna=False).to_string())


# --- (3) Build the patched copy (df_class untouched) ---
df_class_fixed = df_class.copy()
patch_mask = missing_uom_mask & (~df_class_fixed[ut_col].isin(NON_VOLUMETRIC_UTS))
skip_mask  = missing_uom_mask & ( df_class_fixed[ut_col].isin(NON_VOLUMETRIC_UTS))

raw_cap = pd.to_numeric(df_class_fixed.loc[patch_mask, 'CAP_OFFLINE'], errors='coerce')
# BBL/d → kbd (CONVERSION_FACTORS['BBL/d'] == 1/1000)
df_class_fixed.loc[patch_mask, 'CAP_OFFLINE_STD'] = raw_cap / 1000

print(f"\nPatched : {int(patch_mask.sum())} rows (missing UOM, volumetric unit type → BBL/d default)")
print(f"Skipped : {int(skip_mask.sum())} rows (missing UOM, non-volumetric unit type — left as-is)")


# --- (4) Before/after monthly_capacity grid for the FCCU/RFCCU filter ---
grid_fixed = {y: monthly_capacity(df_class_fixed, y, unit_types=UTS) for y in YEARS}
chart_grid_fixed = pd.DataFrame(grid_fixed, index=months).T
chart_grid_fixed.index.name = 'Year'

print("\n=== Monthly avg offline capacity (mb/d) — FCCU + RFCCU ===")
print("\nBEFORE (current bug):")
display(chart_grid.round(2))
print("AFTER (proposed fix):")
display(chart_grid_fixed.round(2))
print("Δ removed by fix:")
display((chart_grid - chart_grid_fixed).round(2))

print(f"\nWorst cell BEFORE : {chart_grid.values.max():.1f} mb/d")
print(f"Worst cell AFTER  : {chart_grid_fixed.values.max():.1f} mb/d")


# --- (5) Side-by-side seasonal charts using the real chart builder ---
seasonal_config = {
    'unit_types': UTS, 'region': None, 'country': None,
    'hist_years': [2021, 2022, 2023, 2024, 2025],
    'exclude_years': [2020],
    'show_years': [],
    'event_type': None,                  # Total (planned + unplanned), matches user's screenshot
}
fig_before = build_seasonal_chart(df_class,       seasonal_config, TODAY)
fig_after  = build_seasonal_chart(df_class_fixed, seasonal_config, TODAY)
fig_before.update_layout(title_text="<b>BEFORE FIX</b> — FCCU + RFCCU Total Maintenance",
                          height=420)
fig_after.update_layout(title_text="<b>AFTER FIX</b> — FCCU + RFCCU Total Maintenance",
                          height=420)
fig_before.show()
fig_after.show()

UNIT_TYPEs that EVER use a non-BBL/d UOM (7):


CAP_UOM,BBL/d,M Lbs/yr,MMSCFD,Metric T/yr,T/d
UTYPE_DESC,,,,,
Amine Treatment,0,0,2206,0,0
Benzene,0,0,0,123,0
Hydrodealkylation,0,0,0,102,0
Olefin/Aromatics Hydrotreater,1501,54,0,0,0
Propylene Splitter,0,575,0,0,0
Refinery Gas/Lights Ends,3553,0,268,0,0
Sulfur Recovery (SRU),0,0,481,0,65495



[REVIEW] UNIT_TYPEs with non-BBL/d UOM that are NOT in your exclusion list:
         {'Olefin/Aromatics Hydrotreater', 'Refinery Gas/Lights Ends', 'Benzene', 'Amine Treatment'}
         → add them to NON_VOLUMETRIC_UTS before applying the real fix.

Events in df_class with missing UOM: 25195


,EVENT_ID,PLANT_NAME,UNIT_NAME,UTYPE_DESC,CAP_OFFLINE,CAP_OFFLINE_STD,U_CAPACITY,U_CAPACITY_STD,START_DATE,END_DATE,E_STATUS_RESEARCH
7,35240,Alliance Refinery,Amine 591 (DEA),Amine Treatment,None,NaN,None,NaN,2015-01-03,2015-03-15,Past
9,38417,Amuay Refinery,Vacuum 3,Vacuum Distillation,None,NaN,None,NaN,2015-04-29,2015-06-02,Past
11,40722,Alliance Refinery,SRU 591,Sulfur Recovery (SRU),None,NaN,None,NaN,2015-01-03,2015-03-15,Past
52,71121,Digboi Refinery,Amine Treatment Unit,Amine Treatment,None,NaN,None,NaN,2015-03-15,2015-04-15,Past
66,77379,El Dorado Refinery (HF Sinclair),AU 3 (Gasifier Amine Stripper),Amine Treatment,None,NaN,None,NaN,2015-03-01,2015-03-15,Past
69,77490,Cheyenne Refinery,Amine Treater (DEA),Amine Treatment,None,NaN,None,NaN,2016-03-28,2016-05-23,Past
79,81261,Esmeraldas Refinery,Visbreaker 2 (Sevia 2),Visbreaker,15000,15000.0,15000,15000.0,2015-04-07,2015-11-30,Past
101,86129,Pasadena Refinery,Amine MDEA,Amine Treatment,None,NaN,None,NaN,2015-01-09,2015-02-07,Cancelled
103,86133,Pasadena Refinery,RSU Reformate Splitter,Reformate Splitter,None,NaN,None,NaN,2015-02-26,2015-04-22,Past
116,86301,Notre-Dame-de-Gravenchon Refinery,Claus 1,Sulfur Recovery (SRU),None,NaN,None,NaN,2015-02-16,2015-04-11,Past



Distribution of those rows by UNIT_TYPE:
UTYPE_DESC
Refinery Gas/Lights Ends                   7420
Amine Treatment                            6560
Sulfur Recovery (SRU)                      3102
Distillate Hydrotreater                    1157
Aromatics                                  1000
Propylene Splitter                          955
Asphalt                                     833
Vacuum Distillation                         825
Lube Oil Hydrotreater                       723
Naphtha Splitter                            626
Light Naphtha Hydrotreater                  370
Isomerization                               221
Hydrodealkylation                           213
Olefin/Aromatics Hydrotreater               158
Diesel Hydrotreater                         157
Reformate Splitter                          116
Visbreaker                                  107
Reformer Feed Hydrotreater                   99
HGO (Heavy Gas Oils) Hydrotreater            87
Mid Distillate Hydrotreater        

...

AFTER (proposed fix):


,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Year,,,,,,,,,,,,
2021,3.10,3.83,3.66,3.36,3.52,3.14,2.08,1.88,2.42,2.36,1.67,1.17
2022,1.60,1.73,1.93,2.23,2.06,2.01,1.75,1.47,1.71,2.33,1.71,1.43
2023,1.38,1.72,1.97,1.89,1.54,1.65,1.19,1.10,1.59,2.38,1.70,0.93
2024,1.21,1.91,1.57,1.91,1.74,1.65,1.63,1.39,1.81,1.73,1.21,0.71
2025,0.97,1.40,1.56,2.31,2.31,1.76,1.59,1.62,2.12,2.29,2.06,1.37
2026,1.19,1.43,1.44,1.87,1.51,1.03,0.75,0.49,0.96,1.10,0.57,0.23


Δ removed by fix:


,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Year,,,,,,,,,,,,
2021,0.0,7.14,19.98,13.32,16.78,15.14,0.00,0.00,0.00,0.0,0.0,0.0
2022,0.0,0.00,36.09,55.94,18.05,0.00,0.00,0.00,0.00,0.0,0.0,0.0
2023,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0
2024,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0
2025,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0
2026,0.0,0.00,0.00,0.00,0.00,0.00,21.66,55.94,9.32,0.0,0.0,0.0



Worst cell BEFORE : 58.2 mb/d
Worst cell AFTER  : 3.8 mb/d


## Cell 18 — Verdict (to be filled in after running)

After running the cells above, summarise here:

| Hypothesis | Verdict | mb/d explained |
|---|---|---|
| A — filter over-match | _PASS / FAIL_ | _0_ |
| B — mixed UOM | _PASS / FAIL_ | _?_ |
| C — open-ended / multi-year events | _PASS / FAIL_ | _?_ |
| D — duplicate EVENT_IDs at same (plant, unit) | _PASS / FAIL_ | _?_ |
| E — `CAP_OFFLINE_STD` > nameplate | _PASS / FAIL_ | _?_ |

**Likely fix** (based on the dominant hypothesis):

- If **C** is the cause → cap each event's overlap window to its own `END_DATE`
  rather than to `yr_end`, **and** drop events with missing `END_DATE` from
  historical-band years (treat missing end as "ongoing today" only).
  Change is local to `monthly_capacity()` in [`charts.py`](charts.py) and the same
  logic in `build_daily_ops_chart()`.

- If **D** is the cause → deduplicate at `(PLANT_NAME, UNIT_NAME, day)` granularity
  inside `monthly_capacity()` before summing, taking max (not sum) of overlapping
  events on the same unit.

- If **B** is the cause → reject non-`BBL/d` rows for FCCU/RFCCU in the chart, or
  fix the UOM upstream.

- If **E** is the cause → cap each event's `CAP_OFFLINE_STD` at its own
  `U_CAPACITY_STD` inside `monthly_capacity()`, or flag/skip the bad rows upstream.

We'll open a separate plan to apply whichever fix is required once the verdict is
known.

---

**Re-run instructions:** restart the kernel and run all cells.  The CSV saved by
Cell 15 (`diagnostics/fccu_top_contributors_<date>.csv`) is the deliverable for
sharing with the IIR data owner.